# 06. Catalog Representation and Dimensionality

    Objective: move from raw catalog rows to meaningful representations. This notebook builds numeric, categorical, text, and interaction-inspired features, then compares PCA and SVD.

In [ ]:
from pathlib import Path
from collections import Counter
import os
import json
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
BUILD_DIR = DATA_DIR / "build"
PLOT_DIR = BASE_DIR / "artifacts" / "plots"

ANIME_PATH = PROCESSED_DIR / "anime_dataset.csv"
RATINGS_PATH = PROCESSED_DIR / "current_user_ratings.csv"

def split_pipe(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    return [part.strip() for part in str(value).split("|") if part.strip()]

def count_pipe(value):
    return len(split_pipe(value))

def explode_pipe_counts(series):
    counts = Counter()
    for value in series.dropna():
        counts.update(split_pipe(value))
    return pd.DataFrame(counts.most_common(), columns=["value", "count"])

def parse_duration_minutes(value):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    if isinstance(value, (int, float)) and not pd.isna(value):
        return float(value) if float(value) > 0 else np.nan

    text = str(value).strip().lower()
    if re.fullmatch(r"\d+(?:\.\d+)?", text):
        numeric = float(text)
        return numeric if numeric > 0 else np.nan

    hours = re.search(r"(\d+(?:\.\d+)?)\s*(?:hr|hour)", text)
    minutes = re.search(r"(\d+(?:\.\d+)?)\s*min", text)
    seconds = re.search(r"(\d+(?:\.\d+)?)\s*sec", text)
    total = 0.0
    if hours:
        total += float(hours.group(1)) * 60
    if minutes:
        total += float(minutes.group(1))
    if seconds:
        total += float(seconds.group(1)) / 60
    return total if total > 0 else np.nan

def infer_season(month):
    if pd.isna(month):
        return np.nan
    month = int(month)
    if month in [1, 2, 3]:
        return "winter"
    if month in [4, 5, 6]:
        return "spring"
    if month in [7, 8, 9]:
        return "summer"
    if month in [10, 11, 12]:
        return "fall"
    return np.nan

def add_bar_labels(ax, fmt="{:.0f}"):
    for patch in ax.patches:
        width = patch.get_width()
        ax.text(width, patch.get_y() + patch.get_height() / 2, " " + fmt.format(width), va="center", fontsize=9)

def save_current_plot(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")

from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.metrics import mean_squared_error

REPRESENTATION_PLOT_DIR = PLOT_DIR / "representation"
REPRESENTATION_PLOT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ANIME_PATH)
df = df.rename(columns={"explicit_genres": "explicit_tags", "explicit_genre_weights": "explicit_tag_weights"})
df["duration_minutes"] = df["duration"].apply(parse_duration_minutes)
df["season_final"] = df["season"].fillna(df["aired_month"].apply(infer_season))
df["total_watch_minutes"] = df["episodes"] * df["duration_minutes"]

for col in ["genres", "tags", "explicit_tags", "demographics", "studios", "season_final", "type", "source", "rating"]:
    if col not in df:
        df[col] = ""
    df[col] = df[col].fillna("")

df.shape


## Representation Design

    We use three complementary views:

    - Numeric popularity/quality/runtime features: useful but biased toward mainstream titles.
    - Multi-hot categorical features: interpretable genres, demographics, type, rating, and season.
    - Text/tag TF-IDF: sparse lexical representation from genres, tags, explicit tags, studios, and demographics.

    PCA is applied to the full anime row set through a dense mixed catalog matrix with numeric, categorical, and capped TF-IDF features. For the larger sparse matrix, Truncated SVD is the appropriate PCA-like method because it works directly on sparse data without forcing a large dense conversion.


In [ ]:
numeric_cols = ["score", "scored_by", "rank", "popularity", "members", "favorites", "episodes", "duration_minutes", "total_watch_minutes", "aired_year", "aired_month"]
numeric_cols = [col for col in numeric_cols if col in df.columns]
numeric_df = df[numeric_cols].fillna(0)
numeric_scaled = StandardScaler().fit_transform(numeric_df)

def lists_from_pipe(series):
    return series.fillna("").apply(split_pipe).tolist()

cat_frames = []
cat_specs = {
    "genres": lists_from_pipe(df["genres"]),
    "demographics": lists_from_pipe(df["demographics"]),
    "type": df["type"].fillna("").apply(lambda x: [x] if x else []).tolist(),
    "rating": df["rating"].fillna("").apply(lambda x: [x] if x else []).tolist(),
    "season": df["season_final"].fillna("").apply(lambda x: [x] if x else []).tolist(),
}

cat_matrices = []
cat_names = []
for name, values in cat_specs.items():
    mlb = MultiLabelBinarizer()
    matrix = mlb.fit_transform(values)
    cat_matrices.append(sparse.csr_matrix(matrix))
    cat_names.extend([f"{name}={cls}" for cls in mlb.classes_])

text = (
    df["genres"].fillna("") + " " +
    df["tags"].fillna("") + " " +
    df["explicit_tags"].fillna("") + " " +
    df["studios"].fillna("") + " " +
    df["demographics"].fillna("")
)

tfidf_small = TfidfVectorizer(max_features=350, min_df=3, max_df=0.85, stop_words="english")
X_text_small = tfidf_small.fit_transform(text)

tfidf_large = TfidfVectorizer(max_features=4000, min_df=3, max_df=0.85, stop_words="english")
X_text_large = tfidf_large.fit_transform(text)

X_numeric = sparse.csr_matrix(numeric_scaled)
X_cat = sparse.hstack(cat_matrices, format="csr")
X_dense_mixed = sparse.hstack([X_numeric, X_cat, X_text_small], format="csr").toarray()
X_sparse_full = sparse.hstack([X_numeric, X_cat, X_text_large], format="csr")

feature_summary = pd.DataFrame([
    {"matrix": "numeric_scaled", "rows": X_numeric.shape[0], "columns": X_numeric.shape[1], "purpose": "score, popularity, runtime, dates"},
    {"matrix": "categorical_multi_hot", "rows": X_cat.shape[0], "columns": X_cat.shape[1], "purpose": "genres, demographics, type, rating, season"},
    {"matrix": "dense_mixed_for_pca", "rows": X_dense_mixed.shape[0], "columns": X_dense_mixed.shape[1], "purpose": "entire catalog PCA with limited TF-IDF"},
    {"matrix": "sparse_full_for_svd", "rows": X_sparse_full.shape[0], "columns": X_sparse_full.shape[1], "purpose": "larger sparse representation for SVD"},
])
feature_summary

## PCA on the Entire Mixed Catalog Matrix

    PCA optimizes variance preservation under orthogonal linear components. It requires dense input in scikit-learn, so the TF-IDF vocabulary is deliberately capped. This is a justified working subset of features, not a different row subset.

In [ ]:
pca = PCA(n_components=min(80, X_dense_mixed.shape[1]), random_state=42)
X_pca = pca.fit_transform(X_dense_mixed)
pca_cum = np.cumsum(pca.explained_variance_ratio_)

thresholds = [0.5, 0.7, 0.8, 0.9, 0.95]
pca_table = []
for threshold in thresholds:
    reached = np.where(pca_cum >= threshold)[0]
    pca_table.append({
        "method": "PCA dense mixed catalog",
        "target_retained_variance": threshold,
        "components_needed": int(reached[0] + 1) if len(reached) else None,
        "max_components_tested": len(pca_cum),
        "retained_variance_at_max": float(pca_cum[-1]),
    })
pca_table = pd.DataFrame(pca_table)
pca_table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(1, len(pca_cum) + 1), pca_cum, marker="o", linewidth=2)
ax.axhline(0.8, color="#E15759", linestyle="--", label="80%")
ax.axhline(0.9, color="#B07AA1", linestyle="--", label="90%")
ax.set_title("PCA on Full Mixed Catalog Feature Matrix")
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative explained variance")
ax.grid(alpha=0.3)
ax.legend()
save_current_plot(REPRESENTATION_PLOT_DIR / "pca_mixed_catalog_explained_variance.png")
plt.show()

### PCA Projection Check

    This plot is a visual diagnostic, not proof of clusters. It checks whether the first two PCA axes are dominated by format/type effects, which helps interpret what PCA is preserving.


In [ ]:
type_codes, type_names = pd.factorize(df["type"].fillna("Unknown"))
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=type_codes, cmap="tab10", s=8, alpha=0.45, linewidths=0)
handles, _ = scatter.legend_elements(num=len(type_names))
ax.legend(handles, type_names, title="type", loc="best", fontsize=8)
ax.set_title("PCA Projection of Dense Mixed Catalog Matrix")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(alpha=0.2)
save_current_plot(REPRESENTATION_PLOT_DIR / "pca_2d_by_type.png")
plt.show()


## Truncated SVD on the Larger Sparse Matrix

    SVD is used for the larger sparse feature space because it avoids densifying thousands of TF-IDF columns. It is still a linear dimensionality reduction method and is appropriate for sparse text vectors.

In [ ]:
n_svd = min(120, X_sparse_full.shape[1] - 1)
svd = TruncatedSVD(n_components=n_svd, random_state=42)
X_svd = svd.fit_transform(X_sparse_full)
svd_cum = np.cumsum(svd.explained_variance_ratio_)

svd_table = []
for threshold in thresholds:
    reached = np.where(svd_cum >= threshold)[0]
    svd_table.append({
        "method": "TruncatedSVD sparse full catalog",
        "target_retained_energy": threshold,
        "components_needed": int(reached[0] + 1) if len(reached) else None,
        "max_components_tested": len(svd_cum),
        "retained_energy_at_max": float(svd_cum[-1]),
    })
svd_table = pd.DataFrame(svd_table)
svd_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(np.arange(1, len(svd_cum) + 1), svd_cum, color="#4C78A8", linewidth=2)
axes[0].set_title("SVD Retained Energy")
axes[0].set_xlabel("Components")
axes[0].set_ylabel("Cumulative explained variance ratio")
axes[0].grid(alpha=0.3)
axes[1].plot(np.arange(1, len(svd.singular_values_) + 1), svd.singular_values_, color="#F28E2B", linewidth=2)
axes[1].set_title("Singular Value Decay")
axes[1].set_xlabel("Component")
axes[1].set_ylabel("Singular value")
axes[1].grid(alpha=0.3)
save_current_plot(REPRESENTATION_PLOT_DIR / "svd_sparse_catalog_energy.png")
plt.show()

### SVD Projection Check

    SVD preserves sparse text/tag structure better than dense PCA. Coloring by member count reveals whether the first two latent axes are partly popularity-driven.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
members_log = np.log1p(pd.to_numeric(df["members"], errors="coerce").fillna(0))
scatter = ax.scatter(X_svd[:, 0], X_svd[:, 1], c=members_log, cmap="viridis", s=8, alpha=0.45, linewidths=0)
fig.colorbar(scatter, ax=ax, label="log(1 + members)")
ax.set_title("SVD Projection of Sparse Text/Tag Catalog Matrix")
ax.set_xlabel("SVD1")
ax.set_ylabel("SVD2")
ax.grid(alpha=0.2)
save_current_plot(REPRESENTATION_PLOT_DIR / "svd_2d_by_members.png")
plt.show()


## Comparison and Interpretation

    PCA on the dense mixed catalog matrix directly answers the full-dataset question, but it uses a smaller TF-IDF vocabulary to stay computationally reasonable. SVD handles the larger sparse representation and is better aligned with text/tag features. t-SNE is included only as an exploratory visualization baseline. It is not used as a production feature space or as cluster proof.


## t-SNE Visualization Baseline

    t-SNE is added for comparison because it is useful for local-neighborhood visualization. It should not be interpreted as compression quality or cluster validation, because it can exaggerate visual separation depending on perplexity, learning rate, and sample choice.


In [ ]:
sample_n = min(6000, len(df))
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(df), size=sample_n, replace=False)

tsne = TSNE(
    n_components=2,
    perplexity=35,
    learning_rate="auto",
    init="pca",
    random_state=42,
)
X_tsne = tsne.fit_transform(X_svd[sample_idx, : min(50, X_svd.shape[1])])

sample_types = df.iloc[sample_idx]["type"].fillna("Unknown")
type_codes, type_names = pd.factorize(sample_types)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=type_codes, cmap="tab10", s=10, alpha=0.6, linewidths=0)
handles, _ = scatter.legend_elements(num=len(type_names))
ax.legend(handles, type_names, title="type", loc="best", fontsize=8)
ax.set_title("t-SNE on SVD Catalog Embeddings (Sample)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.grid(alpha=0.2)
save_current_plot(REPRESENTATION_PLOT_DIR / "tsne_svd_catalog_sample.png")
plt.show()

pd.DataFrame({
    "method": ["t-SNE"],
    "sample_size": [sample_n],
    "input_space": ["first 50 SVD components"],
    "purpose": ["local visualization only, not production compression"],
})


In [ ]:
comparison = pd.concat([
    pca_table.rename(columns={"target_retained_variance": "target", "retained_variance_at_max": "retained_at_max"}),
    svd_table.rename(columns={"target_retained_energy": "target", "retained_energy_at_max": "retained_at_max"}),
], ignore_index=True)
comparison[["method", "target", "components_needed", "max_components_tested", "retained_at_max"]]

## Technical Interpretation

    - Numeric-only representations are simple and interpretable but miss story/content meaning.
    - Text/tag features add content semantics but become sparse and high-dimensional.
    - PCA preserves variance, not semantic similarity. A high-variance axis can still be popularity or format-driven.
    - SVD is more appropriate for the large sparse lexical representation.
    - Visual separation in reduced spaces should not be treated as proof of true clusters; segmentation validates clusters with sweeps and profile analysis.